In [2]:
import pandas as pd
import cloudscraper
import requests
import json
from datetime import date, timedelta
from io import BytesIO

In [13]:
# Automatizar o download dos dados do investing (cotação do dolar e petroleo)

url_base = "https://api.investing.com/api/financialdata/historical"
id_dolar = 2103
id_petroleo = 8833

scraper = cloudscraper.create_scraper(
    browser={
        'browser': 'firefox',
        'platform': 'linux',
        'mobile': False
    }
)

header_request = {
    'domain-id': 'br',
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:128.0) Gecko/20100101 Firefox/128.0',
    'Accept': '*/*',
    'Accept-Language': 'pt-BR,pt;q=0.8,en-US;q=0.5,en;q=0.3',
    'Referer': 'https://br.investing.com/',
    'Origin': 'https://br.investing.com',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-site'
}

scraper.headers.update(header_request)
 
# Intervalo de datas
data_atual = date.today()
data_inicio = data_atual - timedelta(days=365)

In [ ]:
# montar url para baixar cotação petroleo
url = f"{url_base}/{id_petroleo}"

params_request = {
    'start-date': data_inicio,
    'end-date': data_atual,
    'time-frame': 'Weekly',
    'add-missing-rows': 'false'
}

response = scraper.get(url, params=params_request, timeout=30)

dados = response.json()

df_petroleo = pd.DataFrame(dados['data'])

colunas_uteis = {
    'rowDate': 'Data',
    'last_close': 'Fechamento',
    'last_open': 'Abertura',
    'volumeRaw': 'Volume'
}

colunas_finais_petroleo = []
for col_orig, col_nova in colunas_uteis.items():
    if(col_orig in df_petroleo.columns):
        df_petroleo[col_nova] = df_petroleo[col_orig]
        colunas_finais_petroleo.append(col_nova)

df_petroleo = df_petroleo[colunas_finais_petroleo]

print(df_petroleo)


In [ ]:
# montar url para baixar cotação dolar
url = f"{url_base}/{id_dolar}"

params_request = {
    'start-date': data_inicio,
    'end-date': data_atual,
    'time-frame': 'Weekly',
    'add-missing-rows': 'false'
}

response = scraper.get(url, params=params_request, timeout=30)

dados = response.json()

df_dolar = pd.DataFrame(dados['data'])

del colunas_uteis['volumeRaw']

colunas_finais_dolar = []
for col_orig, col_nova in colunas_uteis.items():
    if(col_orig in df_dolar.columns):
        df_dolar[col_nova] = df_dolar[col_orig]
        colunas_finais_dolar.append(col_nova)

df_dolar = df_dolar[colunas_finais_dolar]

print(df_dolar)


In [ ]:
# Automatizar o download dos dados da ANP (preço dos combustíveis)
hoje = date.today()
dias_ate_domingo = (hoje.weekday() + 1) % 7
ultimo_domingo_valido = hoje - timedelta(days=dias_ate_domingo + 14)
ultimo_sabado_valido = ultimo_domingo_valido + timedelta(days=6)
# Formata datas
inicio_str = ultimo_domingo_valido
fim_str = ultimo_sabado_valido
data_final = hoje - timedelta(days= 20)
df_combustiveis = pd.DataFrame()

while inicio_str >= data_final:
    url = f"https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/arquivos-lpc/{inicio_str.year}/revendas_lpc_{inicio_str}_{fim_str}.xlsx"
    
    response = requests.get(url)
    response.raise_for_status()
    xlsx_bytes = BytesIO(response.content)

    df = pd.read_excel(xlsx_bytes)

    # --- AQUI ESTÁ A CORREÇÃO ---
    df.columns = df.iloc[8]   # pega cabeçalho real
    df = df.iloc[9:]          # remove linhas acima
    df = df.reset_index(drop=True)

    df_combustiveis = pd.concat([df_combustiveis, df], ignore_index=True)

    inicio_str -= timedelta(days=7)
    fim_str -= timedelta(days=7)

colunas_uteis = {
    'CNPJ': 'cnpj',
    'RAZÃO': 'razão_social',
    'FANTASIA': 'nome_fantasia',
    'ESTADO': 'estado',
    'MUNICÍPIO': 'municipio',
    'PRODUTO': 'produto',
    'PREÇO DE REVENDA': 'preco',
    'DATA DA COLETA': 'data_coleta'
}

colunas_finais_combustiveis = []
for col_orig, col_nova in colunas_uteis.items():
    if(col_orig in df_combustiveis.columns):
        df_combustiveis[col_nova] = df_combustiveis[col_orig]
        colunas_finais_combustiveis.append(col_nova)

df_combustiveis = df_combustiveis[colunas_finais_combustiveis]

df_combustiveis.to_csv(f'arquivo_{inicio_str}_{fim_str}.csv', index=False, header=True, sep=';')

print(df_combustiveis)